# Buffer Overflow Impact Analysis: GeoBinV4 (100 GB vs 10 TB Buffer)
Comparing two simulation runs to determine if buffer overflow was hurting coverage metrics.

In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
import io
import math
from pathlib import Path

# ── Base paths ──
BASE = Path("/Users/chrischeshire/github/cote-constellation/examples/bent-pipe-constellation/results")

OLD_RUN = BASE / "maxdownload_20260218_204617"
NEW_RUN = BASE / "maxdownload_20260219_072342"

OLD_28MB = OLD_RUN / "constellation_analysis_20260218_214508_28000_200"
NEW_28MB = NEW_RUN / "constellation_analysis_20260219_082538_28000_200"
NEW_2_8MB = NEW_RUN / "constellation_analysis_20260219_072406_02799_200"

SPACINGS = ["close-spaced", "orbit-spaced"]
BIN_SIZE = 5

# CSV paths inside the zip
VIS_LOG_PATH = "geobinv4/visibility_log.csv"
IMG_COMP_PATH = "geobinv4/image_completions.csv"

print("Paths configured ✓")
for label, p in [("Old 28MB", OLD_28MB), ("New 28MB", NEW_28MB), ("New 2.8MB", NEW_2_8MB)]:
    for s in SPACINGS:
        zp = p / s / "simulation_logs.zip"
        print(f"  {label} {s}: exists={zp.exists()}")

Paths configured ✓
  Old 28MB close-spaced: exists=True
  Old 28MB orbit-spaced: exists=True
  New 28MB close-spaced: exists=True
  New 28MB orbit-spaced: exists=True
  New 2.8MB close-spaced: exists=True
  New 2.8MB orbit-spaced: exists=True


In [2]:
def read_csv_from_zip(zip_path, internal_path):
    """Read a CSV from inside a zip file, return DataFrame or None."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        names = zf.namelist()
        # Try exact match first, then search
        if internal_path in names:
            with zf.open(internal_path) as f:
                return pd.read_csv(io.BytesIO(f.read()))
        # Try matching just the tail
        for n in names:
            if n.endswith(internal_path) or internal_path in n:
                with zf.open(n) as f:
                    return pd.read_csv(io.BytesIO(f.read()))
        print(f"  WARNING: '{internal_path}' not found in {zip_path}")
        print(f"  Available: {[n for n in names if n.endswith('.csv')]}")
        return None

def load_data(analysis_dir, spacing):
    """Load both CSVs for a given analysis dir and spacing type."""
    zip_path = analysis_dir / spacing / "simulation_logs.zip"
    vis = read_csv_from_zip(zip_path, VIS_LOG_PATH)
    img = read_csv_from_zip(zip_path, IMG_COMP_PATH)
    return vis, img

def compute_metrics(img_df, vis_df, bin_size=5):
    """Compute all coverage metrics."""
    # 1. Total downloads
    total_downloads = len(img_df)
    
    # 2. Unique 5° bins covered by downloads
    img_df = img_df.copy()
    img_df['lat_bin'] = np.floor(img_df['lat'] / bin_size) * bin_size
    img_df['lon_bin'] = np.floor(img_df['lon'] / bin_size) * bin_size
    download_bins = img_df.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')
    unique_download_bins = len(download_bins)
    
    # 3. Total images captured
    total_captured = int(vis_df['image_taken'].sum()) if 'image_taken' in vis_df.columns else 0
    
    # 4. Unique bins with captures
    vis_captures = vis_df[vis_df['image_taken'] == 1].copy()
    vis_captures['lat_bin'] = np.floor(vis_captures['lat'] / bin_size) * bin_size
    vis_captures['lon_bin'] = np.floor(vis_captures['lon'] / bin_size) * bin_size
    unique_capture_bins = vis_captures.groupby(['lat_bin', 'lon_bin']).ngroups
    
    # 5. Coverage = download bins / capture bins
    coverage = unique_download_bins / unique_capture_bins if unique_capture_bins > 0 else 0
    
    # 6. Max per bin
    max_per_bin = download_bins['count'].max()
    
    # 7. CV of non-zero bin counts
    nonzero_counts = download_bins['count'].values
    cv = nonzero_counts.std() / nonzero_counts.mean() if nonzero_counts.mean() > 0 else 0
    
    return {
        'Total Downloads': total_downloads,
        'Unique Download Bins': unique_download_bins,
        'Total Captured': total_captured,
        'Unique Capture Bins': unique_capture_bins,
        'Coverage': coverage,
        'Max Per Bin': max_per_bin,
        'CV': cv,
        '_bin_counts': download_bins  # for later visualization
    }

print("Helper functions defined ✓")

Helper functions defined ✓


In [3]:
# ── Load all data and peek at columns ──
print("=" * 60)
print("LOADING DATA FROM ZIP FILES")
print("=" * 60)

# First, let's peek at the column names
zip_path = OLD_28MB / "close-spaced" / "simulation_logs.zip"
with zipfile.ZipFile(zip_path, 'r') as zf:
    print("\nFiles in zip:")
    for n in sorted(zf.namelist()):
        print(f"  {n}")

vis_test, img_test = load_data(OLD_28MB, "close-spaced")
print(f"\nvisibility_log columns: {list(vis_test.columns)}")
print(f"visibility_log shape: {vis_test.shape}")
print(vis_test.head(2))
print(f"\nimage_completions columns: {list(img_test.columns)}")
print(f"image_completions shape: {img_test.shape}")
print(img_test.head(2))

LOADING DATA FROM ZIP FILES

Files in zip:
  configuration/
  configuration/constellation.dat
  configuration/date-time.dat
  configuration/gnd-0000000000.dat
  configuration/num-steps.dat
  configuration/rx-gnd-000000.dat
  configuration/sensor.dat
  configuration/time-step.dat
  configuration/tx-sat-000000.dat
  geobinv4/
  geobinv4/evnt-trigger-time.csv
  geobinv4/image_completions.csv
  geobinv4/meas-MB-buffered-sat-0060518000.csv
  geobinv4/meas-MB-buffered-sat-0060518001.csv
  geobinv4/meas-MB-buffered-sat-0060518002.csv
  geobinv4/meas-MB-buffered-sat-0060518003.csv
  geobinv4/meas-MB-buffered-sat-0060518004.csv
  geobinv4/meas-MB-buffered-sat-0060518005.csv
  geobinv4/meas-MB-buffered-sat-0060518006.csv
  geobinv4/meas-MB-buffered-sat-0060518007.csv
  geobinv4/meas-MB-buffered-sat-0060518008.csv
  geobinv4/meas-MB-buffered-sat-0060518009.csv
  geobinv4/meas-MB-buffered-sat-0060518010.csv
  geobinv4/meas-MB-buffered-sat-0060518011.csv
  geobinv4/meas-MB-buffered-sat-0060518012.c

/var/folders/0v/5c9vlpns3w5c58y_r464t87r0000gp/T/ipykernel_19526/1627054697.py:8: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.BytesIO(f.read()))



visibility_log columns: ['time', 'sat_id', 'in_view', 'connected', 'buffer_mb', 'downloaded_mb', 'image_taken', 'lat_deg', 'lon_deg', 'freshness_timestamp', 'distance_km', 'elevation_deg', 'decision_interval', 'bitrate_mbps', 'image_completed', 'completed_image_lat', 'completed_image_lon', 'completed_image_timestamp', 'downloaded_image_lat', 'downloaded_image_lon']
visibility_log shape: (1663487, 20)
   time    sat_id  in_view  connected  buffer_mb  downloaded_mb  image_taken  \
0   2.0  60518000        0          0        0.0            0.0            1   
1   2.0  60518001        0          0        0.0            0.0            1   

     lat_deg     lon_deg            freshness_timestamp  distance_km  \
0  70.980644 -235.768995  2025-08-27T13:03:22.698930000          0.0   
1  70.980245 -235.768465  2025-08-27T13:03:22.698930000          0.0   

   elevation_deg  decision_interval  bitrate_mbps  image_completed  \
0            0.0                  0           0.0                0 

In [4]:
# ── Updated compute_metrics with correct column names ──
def compute_metrics(img_df, vis_df, bin_size=5):
    """Compute all coverage metrics using actual column names."""
    # 1. Total downloads
    total_downloads = len(img_df)
    
    # 2. Unique 5° bins covered by downloads (image_completions uses 'lat', 'lon')
    img = img_df.copy()
    img['lat_bin'] = np.floor(img['lat'] / bin_size) * bin_size
    img['lon_bin'] = np.floor(img['lon'] / bin_size) * bin_size
    download_bins = img.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')
    unique_download_bins = len(download_bins)
    
    # 3. Total images captured (visibility_log uses 'lat_deg', 'lon_deg')
    total_captured = int(vis_df['image_taken'].sum())
    
    # 4. Unique bins with captures
    vis_caps = vis_df[vis_df['image_taken'] == 1].copy()
    vis_caps['lat_bin'] = np.floor(vis_caps['lat_deg'] / bin_size) * bin_size
    vis_caps['lon_bin'] = np.floor(vis_caps['lon_deg'] / bin_size) * bin_size
    capture_bins = vis_caps.groupby(['lat_bin', 'lon_bin']).ngroups
    
    # 5. Coverage
    coverage = unique_download_bins / capture_bins if capture_bins > 0 else 0
    
    # 6. Max per bin
    max_per_bin = int(download_bins['count'].max())
    
    # 7. CV of non-zero bin counts
    counts = download_bins['count'].values
    cv = float(counts.std() / counts.mean()) if counts.mean() > 0 else 0
    
    return {
        'Total Downloads': total_downloads,
        'Unique Download Bins': unique_download_bins,
        'Total Captured': total_captured,
        'Unique Capture Bins': capture_bins,
        'Coverage (%)': round(coverage * 100, 2),
        'Max Per Bin': max_per_bin,
        'CV': round(cv, 4),
        '_bin_df': download_bins
    }

# ── Load ALL data ──
data = {}
configs = {
    'Old-28MB (100GB buf)': OLD_28MB,
    'New-28MB (10TB buf)': NEW_28MB,
    'New-2.8MB (10TB buf)': NEW_2_8MB,
}

for label, path in configs.items():
    data[label] = {}
    for sp in SPACINGS:
        vis, img = load_data(path, sp)
        data[label][sp] = {'vis': vis, 'img': img}
        print(f"Loaded {label} / {sp}: vis={vis.shape}, img={img.shape}")

print("\nAll data loaded ✓")

/var/folders/0v/5c9vlpns3w5c58y_r464t87r0000gp/T/ipykernel_19526/1627054697.py:8: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.BytesIO(f.read()))


Loaded Old-28MB (100GB buf) / close-spaced: vis=(1663487, 20), img=(751, 6)
Loaded Old-28MB (100GB buf) / orbit-spaced: vis=(1656131, 20), img=(9274, 6)


/var/folders/0v/5c9vlpns3w5c58y_r464t87r0000gp/T/ipykernel_19526/1627054697.py:8: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.BytesIO(f.read()))


Loaded New-28MB (10TB buf) / close-spaced: vis=(1663487, 20), img=(743, 6)
Loaded New-28MB (10TB buf) / orbit-spaced: vis=(1656131, 20), img=(9294, 6)


/var/folders/0v/5c9vlpns3w5c58y_r464t87r0000gp/T/ipykernel_19526/1627054697.py:8: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.BytesIO(f.read()))


Loaded New-2.8MB (10TB buf) / close-spaced: vis=(1663487, 20), img=(7102, 6)
Loaded New-2.8MB (10TB buf) / orbit-spaced: vis=(1656131, 20), img=(93306, 6)

All data loaded ✓


In [5]:
# ── Check for buffer overflow log files ──
print("=" * 60)
print("BUFFER OVERFLOW LOG FILE SEARCH")
print("=" * 60)

overflow_found = {}
for label, path in configs.items():
    for sp in SPACINGS:
        zip_path = path / sp / "simulation_logs.zip"
        key = f"{label} / {sp}"
        with zipfile.ZipFile(zip_path, 'r') as zf:
            all_files = zf.namelist()
            overflow_files = [f for f in all_files if 'overflow' in f.lower()]
            overflow_found[key] = overflow_files
            if overflow_files:
                print(f"\n🚨 {key}: OVERFLOW FILES FOUND!")
                for of in overflow_files:
                    print(f"   → {of}")
                    with zf.open(of) as f:
                        content = f.read().decode('utf-8', errors='replace')
                        lines = content.strip().split('\n')
                        print(f"     Lines: {len(lines)}")
                        for line in lines[:5]:
                            print(f"     | {line}")
            else:
                print(f"  ✓ {key}: No overflow files found")

BUFFER OVERFLOW LOG FILE SEARCH

🚨 Old-28MB (100GB buf) / close-spaced: OVERFLOW FILES FOUND!
   → geobinv4/meas-buffer-overflow-sat-60518018.csv
     Lines: 3630
     | time,buffer-overflow-sat-60518018,
     | 2025-08-27T16:01:56.698930000,28.000000,
     | 2025-08-27T16:01:59.698930000,56.000000,
     | 2025-08-27T16:02:02.698930000,84.000000,
     | 2025-08-27T16:02:05.698930000,112.000000,
   → geobinv4/meas-buffer-overflow-sat-60518030.csv
     Lines: 3629
     | time,buffer-overflow-sat-60518030,
     | 2025-08-27T16:01:59.698930000,28.000000,
     | 2025-08-27T16:02:02.698930000,56.000000,
     | 2025-08-27T16:02:05.698930000,84.000000,
     | 2025-08-27T16:02:08.698930000,112.000000,
   → geobinv4/meas-buffer-overflow-sat-60518024.csv
     Lines: 3630
     | time,buffer-overflow-sat-60518024,
     | 2025-08-27T16:01:56.698930000,28.000000,
     | 2025-08-27T16:01:59.698930000,56.000000,
     | 2025-08-27T16:02:02.698930000,84.000000,
     | 2025-08-27T16:02:05.698930000,112.00

In [6]:
# ── Overflow summary ──
print("Overflow file summary:")
for key, files in overflow_found.items():
    status = f"FOUND {len(files)} file(s): {files}" if files else "None"
    print(f"  {key}: {status}")

# Also check buffer_mb stats from visibility logs
print("\n" + "=" * 60)
print("BUFFER USAGE STATISTICS (from visibility_log.csv)")
print("=" * 60)
for label in configs:
    for sp in SPACINGS:
        buf = data[label][sp]['vis']['buffer_mb']
        print(f"\n{label} / {sp}:")
        print(f"  Max buffer_mb: {buf.max():.2f}")
        print(f"  Mean buffer_mb: {buf.mean():.2f}")
        print(f"  99th percentile: {buf.quantile(0.99):.2f}")

Overflow file summary:
  Old-28MB (100GB buf) / close-spaced: FOUND 200 file(s): ['geobinv4/meas-buffer-overflow-sat-60518018.csv', 'geobinv4/meas-buffer-overflow-sat-60518030.csv', 'geobinv4/meas-buffer-overflow-sat-60518024.csv', 'geobinv4/meas-buffer-overflow-sat-60518187.csv', 'geobinv4/meas-buffer-overflow-sat-60518193.csv', 'geobinv4/meas-buffer-overflow-sat-60518178.csv', 'geobinv4/meas-buffer-overflow-sat-60518144.csv', 'geobinv4/meas-buffer-overflow-sat-60518150.csv', 'geobinv4/meas-buffer-overflow-sat-60518151.csv', 'geobinv4/meas-buffer-overflow-sat-60518145.csv', 'geobinv4/meas-buffer-overflow-sat-60518179.csv', 'geobinv4/meas-buffer-overflow-sat-60518192.csv', 'geobinv4/meas-buffer-overflow-sat-60518186.csv', 'geobinv4/meas-buffer-overflow-sat-60518025.csv', 'geobinv4/meas-buffer-overflow-sat-60518031.csv', 'geobinv4/meas-buffer-overflow-sat-60518019.csv', 'geobinv4/meas-buffer-overflow-sat-60518027.csv', 'geobinv4/meas-buffer-overflow-sat-60518033.csv', 'geobinv4/meas-buf

In [7]:
# ── Compact overflow + buffer summary ──
overflow_summary = {k: len(v) for k, v in overflow_found.items()}
print("Overflow files found per config:", overflow_summary)

buf_stats = {}
for label in configs:
    for sp in SPACINGS:
        buf = data[label][sp]['vis']['buffer_mb']
        buf_stats[f"{label}|{sp}"] = {'max': buf.max(), 'mean': round(buf.mean(), 2)}
print("\nBuffer MB max/mean:")
for k, v in buf_stats.items():
    print(f"  {k}: max={v['max']:.1f} MB, mean={v['mean']:.1f} MB")

Overflow files found per config: {'Old-28MB (100GB buf) / close-spaced': 200, 'Old-28MB (100GB buf) / orbit-spaced': 200, 'New-28MB (10TB buf) / close-spaced': 0, 'New-28MB (10TB buf) / orbit-spaced': 0, 'New-2.8MB (10TB buf) / close-spaced': 0, 'New-2.8MB (10TB buf) / orbit-spaced': 0}

Buffer MB max/mean:
  Old-28MB (100GB buf)|close-spaced: max=104857.6 MB, mean=77311.2 MB
  Old-28MB (100GB buf)|orbit-spaced: max=104857.6 MB, mean=78652.3 MB
  New-28MB (10TB buf)|close-spaced: max=211363.6 MB, mean=102896.5 MB
  New-28MB (10TB buf)|orbit-spaced: max=210418.0 MB, mean=105015.5 MB
  New-2.8MB (10TB buf)|close-spaced: max=21136.4 MB, mean=10239.8 MB
  New-2.8MB (10TB buf)|orbit-spaced: max=20166.3 MB, mean=9869.8 MB
